# INCLUDE50 CNN+LSTM Training
MobileNetV2 features + BiLSTM — targets 90%+ accuracy

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Clone repo ─────────────────────────────────────────────────────
import os, subprocess
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/Major_Project'], check=False)
!git clone -b branch_03_cnn-and-lstm https://github.com/BishalDubey27/Major_Project.git /content/Major_Project
os.chdir('/content/Major_Project/INCLUDE')
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── 4. Set paths ──────────────────────────────────────────────────────
import os

DATASET_DIR  = '/content/drive/MyDrive/ISL_Training/new_include50_videos'  # extracted videos
KEYPOINTS_DIR = '/content/drive/MyDrive/ISL_Training/keypoints'            # pre-generated keypoints
CNN_DIR      = '/content/drive/MyDrive/ISL_Training/cnn_features'          # CNN features output
SAVE_PATH    = '/content/drive/MyDrive/ISL_Training/trained_model'

os.makedirs(CNN_DIR, exist_ok=True)
os.makedirs(SAVE_PATH, exist_ok=True)

print('Keypoints dir:', KEYPOINTS_DIR)
print('CNN features dir:', CNN_DIR)
print('Save path:', SAVE_PATH)

In [ ]:
# ── 5. Check existing keypoints ───────────────────────────────────────
import glob
for split in ['train', 'val', 'test']:
    files = glob.glob(f'{KEYPOINTS_DIR}/include50_{split}_keypoints/*.json')
    print(f'{split}: {len(files)} keypoint files')

In [ ]:
# ── 6. (Optional) Generate missing keypoints ──────────────────────────
# Skip this cell if you already have enough keypoints from previous runs
import os, sys
sys.path.insert(0, '/content/Major_Project/INCLUDE')
os.chdir('/content/Major_Project/INCLUDE')

from generate_keypoints import process_video, load_file

for split in ['train', 'val', 'test']:
    save_dir = f'{KEYPOINTS_DIR}/include50_{split}_keypoints'
    os.makedirs(save_dir, exist_ok=True)
    paths = load_file(f'train_test_paths/include50_{split}.txt', DATASET_DIR)
    skipped = processed = errors = 0
    for path in paths:
        label = path.replace('\\', '/').split('/')[-2]
        label = ''.join([i for i in label if i.isalpha()]).lower()
        uid = '_'.join([label, os.path.splitext(os.path.basename(path))[0]])
        out_file = os.path.join(save_dir, f'{uid}.json')
        if os.path.exists(out_file):
            skipped += 1; continue
        try:
            process_video(path, save_dir); processed += 1
        except Exception as e:
            errors += 1
    print(f'{split}: {processed} new, {skipped} skipped, {errors} errors')

In [ ]:
# ── 7. Extract CNN features from keypoints ────────────────────────────
# MobileNetV2 renders each skeleton frame and extracts 1280-dim features
import sys, os
sys.path.insert(0, '/content/Major_Project/INCLUDE')
os.chdir('/content/Major_Project/INCLUDE')

import argparse
args = argparse.Namespace(
    dataset='include50',
    data_dir=KEYPOINTS_DIR,
    save_dir=CNN_DIR,
    use_cnn=True,
    use_augs=False,
    model='lstm',
    transformer_size='small',
    seed=0, batch_size=32, epochs=100,
    learning_rate=1e-4,
    save_path=SAVE_PATH
)

from cnn_runner import save_cnn_features
save_cnn_features(args)

import glob
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f'{CNN_DIR}/include50_{split}_cnn_features/*.npy'))
    print(f'{split}: {n} CNN feature files')

In [ ]:
# ── 8. Train CNN+LSTM model ───────────────────────────────────────────
import os
os.chdir('/content/Major_Project/INCLUDE')

# Increase early stopping patience
with open('train_nn.py', 'r') as f:
    content = f.read()
content = content.replace('EarlyStopping(patience=10', 'EarlyStopping(patience=20')
with open('train_nn.py', 'w') as f:
    f.write(content)
print('Patience set to 20')

!python runner.py \
    --dataset include50 \
    --model lstm \
    --use_cnn \
    --data_dir {CNN_DIR} \
    --save_path {SAVE_PATH} \
    --epochs 100 \
    --batch_size 32 \
    --learning_rate 1e-4 \
    --use_augs

In [ ]:
# ── 9. Check saved model ──────────────────────────────────────────────
import torch, glob

pth_files = glob.glob(f'{SAVE_PATH}/*.pth')
print('Saved models:', pth_files)

for pf in pth_files:
    cp = torch.load(pf, map_location='cpu', weights_only=False)
    print(f'{pf}: score={cp.get("score", "N/A")}')
    for k,v in cp['model'].items():
        if 'fc' in k or 'linear' in k or 'out' in k.lower():
            if hasattr(v, 'shape'): print(f'  Output layer: {k} {v.shape}')

In [ ]:
# ── 10. Download the model ────────────────────────────────────────────
from google.colab import files
import glob

pth_files = glob.glob(f'{SAVE_PATH}/*.pth')
# Download the CNN+LSTM model (cnn_lstm.pth or augs_lstm.pth)
cnn_model = [f for f in pth_files if 'lstm' in f.lower()]
if cnn_model:
    files.download(cnn_model[0])
    print('Downloaded:', cnn_model[0])
    print()
    print('Next steps:')
    print('1. Place this .pth file in your INCLUDE/ folder')
    print('2. Tell Kiro the filename to update unified_app.py')
else:
    print('No LSTM model found, downloading first available:')
    files.download(pth_files[0])